# Basic Ordinal Logistic Regression (statsmodels)

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel

# Create sample ordinal data
np.random.seed(42)

df = pd.DataFrame({
    "age": np.random.randint(18, 60, 200),
    "income": np.random.randint(20000, 80000, 200),
})

# Ordinal target (0 < 1 < 2)
df["rating"] = pd.cut(
    df["income"],
    bins=3,
    labels=[0,1,2]
).astype(int)

X = df[["age", "income"]]
y = df["rating"]

# Fit ordinal logistic model
model = OrderedModel(y, X, distr="logit")
result = model.fit(method="bfgs")

print(result.summary())

Optimization terminated successfully.
         Current function value: 0.000000
         Iterations: 239
         Function evaluations: 275
         Gradient evaluations: 275
                             OrderedModel Results                             
Dep. Variable:                 rating   Log-Likelihood:            -4.2102e-08
Model:                   OrderedModel   AIC:                             8.000
Method:            Maximum Likelihood   BIC:                             21.19
Date:                Fri, 13 Mar 2026                                         
Time:                        22:54:09                                         
No. Observations:                 200                                         
Df Residuals:                     196                                         
Df Model:                           2                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------

# Prediction with Ordinal Logistic

In [4]:
# Predict probabilities
pred_prob = result.predict(X)

print("First 5 predictions:")
print(pred_prob[:5])

First 5 predictions:
     0              1    2
0  0.0   1.000000e+00  0.0
1  0.0   1.000000e+00  0.0
2  1.0   0.000000e+00  0.0
3  0.0  7.826662e-106  1.0
4  0.0   0.000000e+00  1.0


# Predicted Class Labels

In [5]:
# Convert probabilities → class
y_pred = pred_prob.idxmax(axis=1)

print(y_pred.head())

0    1
1    1
2    0
3    2
4    2
dtype: int64


# Ordinal Logistic Using mord Library

In [7]:
import mord
from sklearn.model_selection import train_test_split

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Ordinal logistic model
model = mord.LogisticAT()

model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

print("Accuracy:", (y_pred == y_test).mean())

Accuracy: 1.0


# Ordinal Logistic with Standardization

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import mord

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ordinal", mord.LogisticAT())
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
print("Accuracy:", (y_pred == y_test).mean())

Accuracy: 1.0


# Ordinal Logistic with Confusion Matrix

In [9]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[12  0  0]
 [ 0  9  0]
 [ 0  0 19]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        19

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

